# Self-test: is your lab ready for the workshop?

This notebook checks, in four short steps, that your lab has what the workshop needs.
Each step explains what it checks, and its code is short enough to read.

**Before you run it**, check your Git name and email (preparation step 3).

**How to run it:** choose **Run → Run All Cells**. Each check prints `PASS` or `FAIL`.
A `FAIL` also says what to do. Fix it, then run all cells again.

The notebook only reads. It changes nothing except one small test file in `work/`, which
it deletes again.

In [ ]:
# Tools used by the checks below, and a helper that prints one result.
import os
import subprocess
from pathlib import Path

WORK = Path.home() / "work"              # the folder that is kept when the lab stops
results = {}


def report(name, ok, detail, what_to_do):
    results[name] = ok
    print(("PASS" if ok else "FAIL") + f"  {name}: {detail}")
    if not ok:
        print(f"      What to do: {what_to_do}")

## 1. Python and the analysis packages

The workshop uses Python 3.13 with pandas, NumPy, SciPy, xarray and Matplotlib. This
cell loads them and shows their versions.

**Expected:** `PASS  Python and packages: Python 3.13.x with pandas ..., numpy ..., ...`

In [ ]:
import sys

try:
    import matplotlib, numpy, pandas, scipy, xarray
    versions = ", ".join(f"{m.__name__} {m.__version__}" for m in (pandas, numpy, scipy, xarray, matplotlib))
    ok = sys.version_info[:2] == (3, 13)
    detail = f"Python {sys.version.split()[0]} with {versions}"
except ImportError as error:
    ok, detail = False, f"a package is missing ({error.name})"

report("Python and packages", ok, detail,
       "The lab was not started from the workshop page. Stop it (File → Hub Control Panel → "
       "Stop My Server) and start it again from the workshop page.")

## 2. Example data

Reads the example dataset, *E. coli* growth at 37 °C with 1 % glucose, and plots how
the optical density of the culture rises over time.

**Expected:** `PASS  Example data: 289 readings from 8 wells`, followed by a curve that
rises from about 0.14 to about 0.58.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import pandas as pd

    data = pd.read_csv("data/ecoli-growth/ecoli-growth-37c-1pct.csv")
    wells = ["A7", "B7", "C7", "D7", "E7", "F7", "G7", "H7"]
    hours = data["tcultivo (min)"] / 60
    mean_od = data[wells].mean(axis="columns")

    plt.figure(figsize=(6, 3))
    plt.plot(hours, mean_od)
    plt.xlabel("time (h)")
    plt.ylabel("mean optical density")
    plt.title("E. coli, 37 °C, 1 % glucose")
    plt.show()

    report("Example data", len(data) == 289, f"{len(data)} readings from {len(wells)} wells",
           "The example data was changed. Stop the lab and start it again from the workshop page: "
           "the original comes back with every start.")
except (ImportError, OSError, KeyError) as error:
    report("Example data", False, f"cannot read the data ({error})",
           "Open this notebook from your home folder, the one with data/ and template/, and run it again.")

## 3. The `work/` folder is kept

Only `work/` is kept when the lab stops; everything else is reset at the next start.
This cell checks that `work/` is the kept storage and that you can write to it.

**Expected:** `PASS  work folder: /home/jovyan/work is kept storage and writable`

In [ ]:
test_file = WORK / ".self-test.txt"
try:
    test_file.write_text("test")
    test_file.unlink()
    writable = True
except OSError:
    writable = False

kept = os.path.ismount(WORK)
report("work folder", kept and writable,
       f"{WORK} is kept storage and writable" if kept and writable
       else f"{WORK} is " + ("not writable" if kept else "not kept storage"),
       "Stop the lab (File → Hub Control Panel → Stop My Server) and start it again from the "
       "workshop page." if not kept else
       "Your storage may be full. Delete files you no longer need in work/ and run this cell again.")

## 4. Your Git name and email

Git writes your name and email into every commit. This cell shows the values the lab
uses (preparation step 3).

**Expected:** `PASS  Git name and email: <your name> <your email>`

In [ ]:
def git_config(key):
    return subprocess.run(["git", "config", "--global", key], capture_output=True, text=True).stdout.strip()

name, email = git_config("user.name"), git_config("user.email")
report("Git name and email", bool(name and email), f"{name or '(no name)'} <{email or 'no email'}>",
       'Set them in a terminal (File → New → Terminal): git config --global user.name "Your Name" '
       'and git config --global user.email "you@example.org", then run this cell again.')

## Summary

**Expected:** `All 4 checks passed. Your lab is ready for the workshop.`

In [ ]:
failed = [name for name, ok in results.items() if not ok]
if failed:
    print(f"{len(failed)} of {len(results)} checks failed: {', '.join(failed)}.")
    print("Follow 'What to do' under each failed check, then run all cells again.")
else:
    print(f"All {len(results)} checks passed. Your lab is ready for the workshop.")